# PART 1 — Data Loading, 3D Grid Construction, Physics-derived Fields, Sequencing

In [ ]:
import os, math, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.interpolate import griddata
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CSV_PATH = "Oman_HAB_3D_Dataset - run.csv"

RAW_COLS = ["Latitude","Longitude","Date","Depth [m]","Temp. [degC]","Sal.",
            "Current_U [m/s]","Current_V [m/s]","SSH [m]","Chl-a [Âµg/l]",
            "O2 [mmol/m3]","Nitrate [mmol/m3]","Phosphate [mmol/m3]",
            "Primary_Productivity [mg C/m3/day]","N2","DO [mg/l]","MLD",
            "season_sin","season_cos","surface_RHI_frequency",
            "subsurface_RHI_frequency","depth_of_heatwave"]

RENAME = {
    "Latitude":"lat","Longitude":"lon","Date":"date","Depth [m]":"depth",
    "Temp. [degC]":"temp","Sal.":"sal","Current_U [m/s]":"u","Current_V [m/s]":"v",
    "SSH [m]":"ssh","Chl-a [Âµg/l]":"chl","O2 [mmol/m3]":"o2",
    "Nitrate [mmol/m3]":"no3","Phosphate [mmol/m3]":"po4",
    "Primary_Productivity [mg C/m3/day]":"pp","N2":"n2","DO [mg/l]":"do_",
    "MLD":"mld","season_sin":"season_sin","season_cos":"season_cos",
    "surface_RHI_frequency":"srhi","subsurface_RHI_frequency":"subrhi",
    "depth_of_heatwave":"hwdepth"
}

def load_raw_dataframe(path=CSV_PATH):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    fixed = {}
    for c in df.columns:
        for raw in RAW_COLS:
            if c == raw or c.replace("Âµ","µ") == raw.replace("Âµ","µ"):
                fixed[c] = RENAME[raw]
                break
    df = df.rename(columns=fixed)
    needed = list(RENAME.values())
    for col in needed:
        if col not in df.columns:
            df[col] = np.nan
    df = df[needed].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["lat","lon","date","depth"])
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["doy"] = df["date"].dt.dayofyear
    for c in ["chl","temp","sal","u","v","ssh","o2","no3","po4","pp","n2","do_",
              "mld","season_sin","season_cos","srhi","subrhi","hwdepth"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

class GridConfig:
    def __init__(self, df, n_lat=24, n_lon=24, n_depth=12,
                 seq_len=10, horizons=(1,3,7)):
        self.lat_edges = np.linspace(df["lat"].min(), df["lat"].max(), n_lat+1)
        self.lon_edges = np.linspace(df["lon"].min(), df["lon"].max(), n_lon+1)
        self.lat_centers = (self.lat_edges[:-1]+self.lat_edges[1:])/2
        self.lon_centers = (self.lon_edges[:-1]+self.lon_edges[1:])/2
        self.depth_levels = np.quantile(df["depth"].dropna().unique(),
                                         np.linspace(0,1,n_depth))
        self.depth_levels = np.unique(np.round(self.depth_levels,2))
        self.n_depth = len(self.depth_levels)
        self.n_lat = n_lat
        self.n_lon = n_lon
        self.seq_len = seq_len
        self.horizons = horizons

FEATURE_VARS = ["temp","sal","u","v","ssh","o2","no3","po4","pp","n2","do_",
                 "mld","season_sin","season_cos","srhi","subrhi","hwdepth","chl"]
TARGET_VAR = "chl"

def build_3d_grid_for_date(df_day, cfg):
    n_lat, n_lon, n_depth = cfg.n_lat, cfg.n_lon, cfg.n_depth
    grid = np.full((len(FEATURE_VARS), n_depth, n_lat, n_lon), np.nan, dtype=np.float32)
    lat_idx = np.clip(np.digitize(df_day["lat"], cfg.lat_edges)-1, 0, n_lat-1)
    lon_idx = np.clip(np.digitize(df_day["lon"], cfg.lon_edges)-1, 0, n_lon-1)
    depth_idx = np.array([np.argmin(np.abs(cfg.depth_levels-d)) for d in df_day["depth"]])
    for vi, var in enumerate(FEATURE_VARS):
        vals = df_day[var].values
        np.add.at(grid[vi], (depth_idx, lat_idx, lon_idx), np.nan_to_num(vals, nan=0.0))
    counts = np.zeros((n_depth, n_lat, n_lon), dtype=np.float32)
    np.add.at(counts, (depth_idx, lat_idx, lon_idx), 1.0)
    counts[counts==0] = np.nan
    for vi in range(len(FEATURE_VARS)):
        grid[vi] = grid[vi] / counts
    for vi in range(len(FEATURE_VARS)):
        for dz in range(n_depth):
            layer = grid[vi, dz]
            mask = ~np.isnan(layer)
            if mask.sum() >= 4:
                pts = np.array(np.nonzero(mask)).T
                vals = layer[mask]
                gy, gx = np.mgrid[0:n_lat, 0:n_lon]
                filled = griddata(pts, vals, (gy, gx), method="linear")
                nanmask = np.isnan(filled)
                if nanmask.any():
                    filled_n = griddata(pts, vals, (gy, gx), method="nearest")
                    filled[nanmask] = filled_n[nanmask]
                grid[vi, dz] = filled
            elif mask.sum() > 0:
                grid[vi, dz] = np.nanmean(vals)
            else:
                grid[vi, dz] = 0.0
    return np.nan_to_num(grid, nan=0.0)

def build_full_spatiotemporal_tensor(df, cfg, cache_path="grid_cache.npz"):
    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=True)
        return z["tensor"], pd.to_datetime(z["dates"])
    dates = sorted(df["date"].dropna().unique())
    tensors = []
    for d in dates:
        df_day = df[df["date"]==d]
        if len(df_day) == 0:
            continue
        g = build_3d_grid_for_date(df_day, cfg)
        tensors.append(g)
    tensor = np.stack(tensors, axis=0)  # [T, C, D, H, W]
    np.savez_compressed(cache_path, tensor=tensor, dates=np.array(dates))
    return tensor, pd.to_datetime(dates)

def diagnose_vertical_velocity(tensor, feat_idx, cfg, dt_seconds=86400.0):
    u = tensor[:, feat_idx["u"]]
    v = tensor[:, feat_idx["v"]]
    dlat = np.gradient(cfg.lat_centers).mean()*111000.0
    dlon = np.gradient(cfg.lon_centers).mean()*111000.0
    dudx = np.gradient(u, axis=-1) / dlon
    dvdy = np.gradient(v, axis=-2) / dlat
    div_h = dudx + dvdy
    depth_levels = cfg.depth_levels
    n_depth = len(depth_levels)
    w = np.zeros_like(u)
    for k in range(1, n_depth):
        dz = max(depth_levels[k]-depth_levels[k-1], 1e-3)
        w[:, k] = w[:, k-1] - div_h[:, k-1]*dz
    return w.astype(np.float32)

def diagnose_vertical_diffusivity(tensor, feat_idx, cfg, k_bg=1e-5, k_mix=1e-2):
    mld = tensor[:, feat_idx["mld"]]
    n2 = tensor[:, feat_idx["n2"]]
    n_depth = len(cfg.depth_levels)
    depth_levels = cfg.depth_levels.reshape(1, n_depth, 1, 1)
    above_mld = depth_levels <= mld[:, None, :, :]
    n2_safe = np.clip(n2, 1e-8, None)
    k_strat = k_bg + k_mix / (1.0 + n2_safe*1e3)
    K = np.where(above_mld, k_mix, k_strat)
    return K.astype(np.float32)

FEAT_IDX = {v:i for i,v in enumerate(FEATURE_VARS)}

def normalize_tensor(tensor):
    T, C, D, H, W = tensor.shape
    flat = tensor.reshape(T, C, -1)
    mean = flat.mean(axis=(0,2), keepdims=True)
    std = flat.std(axis=(0,2), keepdims=True) + 1e-6
    norm = (flat - mean) / std
    return norm.reshape(T, C, D, H, W), mean.reshape(C), std.reshape(C)

def make_sequences(tensor, dates, cfg, target_idx):
    T = tensor.shape[0]
    seq_len = cfg.seq_len
    max_h = max(cfg.horizons)
    X, Y, aux_dates = [], {h:[] for h in cfg.horizons}, []
    for t in range(T - seq_len - max_h + 1):
        X.append(tensor[t:t+seq_len])
        for h in cfg.horizons:
            Y[h].append(tensor[t+seq_len+h-1, target_idx])
        aux_dates.append(dates[t+seq_len-1])
    X = np.stack(X).astype(np.float32)
    for h in cfg.horizons:
        Y[h] = np.stack(Y[h]).astype(np.float32)
    return X, Y, pd.to_datetime(aux_dates)

def leave_one_year_out_split(dates, years_all=None):
    years = pd.to_datetime(dates).year
    uniq = sorted(pd.unique(years))
    for test_year in uniq:
        test_mask = years == test_year
        train_mask = ~test_mask
        yield test_year, train_mask, test_mask

def train_val_test_split_by_time(n, train_frac=0.7, val_frac=0.15):
    idx = np.arange(n)
    n_train = int(n*train_frac)
    n_val = int(n*val_frac)
    return idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

def get_regime_labels(df_or_tensor_dates):
    dates = pd.to_datetime(df_or_tensor_dates)
    month = dates.month
    monsoon = np.isin(month, [6,7,8,9])
    return np.where(monsoon, "monsoon", "inter_monsoon")

print("PART 1 loaded: data pipeline utilities ready.")

if __name__ == "__main__":
    df = load_raw_dataframe(CSV_PATH)
    cfg = GridConfig(df)
    tensor, dates = build_full_spatiotemporal_tensor(df, cfg)
    tensor_norm, feat_mean, feat_std = normalize_tensor(tensor)
    w = diagnose_vertical_velocity(tensor, FEAT_IDX, cfg)
    K = diagnose_vertical_diffusivity(tensor, FEAT_IDX, cfg)
    X, Y, seq_dates = make_sequences(tensor_norm, dates, cfg, FEAT_IDX["chl"])
    tr_idx, va_idx, te_idx = train_val_test_split_by_time(len(X))
    print("Tensor:", tensor.shape, "Sequences:", X.shape)


# PART 2 — Physics Residual (Advection-Diffusion-Reaction) & Dynamic Causal Graph Module

In [ ]:
# ================= PART 2: PHYSICS RESIDUAL + DYNAMIC CAUSAL GRAPH =================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import combinations

# ---------- 2.1 Finite-difference operators (as conv kernels) ----------
def grad_x(field, dx):
    # field: [B, D, H, W]
    kernel = torch.tensor([-1.0, 0.0, 1.0], device=field.device).view(1,1,1,1,3) / (2*dx)
    f = field.unsqueeze(1)
    f = F.pad(f, (1,1,0,0,0,0), mode="replicate")
    return F.conv3d(f, kernel).squeeze(1)

def grad_y(field, dy):
    kernel = torch.tensor([-1.0, 0.0, 1.0], device=field.device).view(1,1,1,3,1) / (2*dy)
    f = field.unsqueeze(1)
    f = F.pad(f, (0,0,1,1,0,0), mode="replicate")
    return F.conv3d(f, kernel).squeeze(1)

def grad_z(field, dz):
    kernel = torch.tensor([-1.0, 0.0, 1.0], device=field.device).view(1,1,3,1,1) / (2*dz)
    f = field.unsqueeze(1)
    f = F.pad(f, (0,0,0,0,1,1), mode="replicate")
    return F.conv3d(f, kernel).squeeze(1)

def laplacian_z_weighted(field, K, dz):
    # d/dz ( K dC/dz )  -- flux form, K: [B,D,H,W]
    dCdz = grad_z(field, dz)
    flux = K * dCdz
    d2 = grad_z(flux, dz)
    return d2

# ---------- 2.2 Michaelis-Menten biological source/sink term ----------
def reaction_term(C, no3, temp, mld=None, depth_levels=None, irradiance=None,
                   mu_max=1.2, k_no3=0.5, k_t=15.0, m_loss=0.15,
                   k_par=0.15, use_light=False):
    no3 = torch.clamp(no3, min=0.0)
    nutrient_lim = no3 / (k_no3 + no3 + 1e-6)
    temp_lim = torch.sigmoid((temp - k_t) / 3.0)
    if use_light and irradiance is not None and depth_levels is not None:
        z = depth_levels.view(1,-1,1,1).to(C.device)
        I_z = irradiance * torch.exp(-k_par * z)
        light_lim = I_z / (I_z + 20.0)
    else:
        light_lim = torch.ones_like(C)
    growth = mu_max * nutrient_lim * temp_lim * light_lim * torch.clamp(C, min=0.0)
    loss = m_loss * torch.clamp(C, min=0.0)
    return growth - loss

# ---------- 2.3 PDE residual: advection-diffusion-reaction ----------
def pde_residual(C_seq, u, v, w, K, no3, temp, dt, dx, dy, dz,
                  mld=None, depth_levels=None, irradiance=None, use_light=False):
    # C_seq: [B, T, D, H, W] predicted chlorophyll sequence
    dCdt = (C_seq[:,1:] - C_seq[:,:-1]) / dt
    C_mid = 0.5*(C_seq[:,1:] + C_seq[:,:-1])
    residuals = []
    for t in range(C_mid.shape[1]):
        Ct = C_mid[:, t]
        adv_x = u[:, t] * grad_x(Ct, dx)
        adv_y = v[:, t] * grad_y(Ct, dy)
        adv_z = w[:, t] * grad_z(Ct, dz)
        diff = laplacian_z_weighted(Ct, K[:, t], dz)
        R = reaction_term(Ct, no3[:, t], temp[:, t], mld[:, t] if mld is not None else None,
                           depth_levels, irradiance, use_light=use_light)
        res_t = dCdt[:, t] + adv_x + adv_y + adv_z - diff - R
        residuals.append(res_t)
    residual = torch.stack(residuals, dim=1)
    return residual

def pde_residual_loss(C_seq, u, v, w, K, no3, temp, dt=1.0, dx=1.0, dy=1.0, dz=1.0,
                       mld=None, depth_levels=None, irradiance=None, use_light=False):
    res = pde_residual(C_seq, u, v, w, K, no3, temp, dt, dx, dy, dz,
                        mld, depth_levels, irradiance, use_light)
    return (res**2).mean()

# ---------- 2.4 Optional stoichiometric coupling (Chl - O2 - Nitrate) ----------
def stoichiometric_residual_loss(C_seq, O2_seq, NO3_seq, redfield_O2C=6.625, redfield_NC=0.15,
                                  mask=None):
    dC = C_seq[:,1:] - C_seq[:,:-1]
    dO2 = O2_seq[:,1:] - O2_seq[:,:-1]
    dNO3 = NO3_seq[:,1:] - NO3_seq[:,:-1]
    res_o2 = dO2 - redfield_O2C*dC
    res_no3 = dNO3 + redfield_NC*dC
    res = res_o2**2 + res_no3**2
    if mask is not None:
        res = res * mask
        return res.sum() / (mask.sum() + 1e-6)
    return res.mean()

# ---------- 2.5 Physics-constrained differentiable optimization layer ----------
class PhysicsProjectionLayer(nn.Module):
    """Soft-constrained proximal layer enforcing non-negativity and
    approximate mass conservation via a few unrolled gradient-projection steps."""
    def __init__(self, n_iter=5, step_size=0.1, mass_weight=1.0):
        super().__init__()
        self.n_iter = n_iter
        self.step_size = step_size
        self.mass_weight = mass_weight

    def forward(self, C_pred, C_prior_mass=None):
        C = C_pred
        for _ in range(self.n_iter):
            C = F.softplus(C)
            if C_prior_mass is not None:
                cur_mass = C.mean(dim=list(range(1, C.dim())), keepdim=True)
                target_mass = C_prior_mass.view(*cur_mass.shape)
                grad_mass = (cur_mass - target_mass)
                C = C - self.step_size * self.mass_weight * grad_mass
        return F.softplus(C)

# ---------- 2.6 Dynamic causal discovery (regime-precomputed, lagged-regression /
#              Granger-style approximation of PCMCI+, updated only per window) ----------
CAUSAL_VARS = ["temp","sal","u","v","ssh","o2","no3","po4","n2","mld","chl"]

def _lag_design_matrix(series_dict, max_lag=3):
    names = list(series_dict.keys())
    T = len(next(iter(series_dict.values())))
    X_list, y_names = [], []
    for lag in range(1, max_lag+1):
        for n in names:
            X_list.append(series_dict[n][:-lag] if lag>0 else series_dict[n])
    return names

def granger_causal_adjacency(regime_series, max_lag=3, p_thresh=0.05):
    """regime_series: dict var_name -> 1D numpy array (spatially averaged time series
    for a regime). Returns adjacency matrix [n_vars, n_vars] with directed edge
    strength cause->effect based on lagged-regression partial R^2 (lightweight
    Granger-style proxy for PCMCI+/DYNOTEARS, feasible at regime granularity)."""
    from numpy.linalg import lstsq
    names = list(regime_series.keys())
    n = len(names)
    T = len(regime_series[names[0]])
    adj = np.zeros((n, n), dtype=np.float32)
    if T <= max_lag + 5:
        return adj
    for i, effect in enumerate(names):
        y = regime_series[effect][max_lag:]
        X_self = np.stack([regime_series[effect][max_lag-l:T-l] for l in range(1,max_lag+1)], axis=1)
        X_self = np.hstack([X_self, np.ones((len(y),1))])
        beta_self, *_ = lstsq(X_self, y, rcond=None)
        resid_self = y - X_self @ beta_self
        ss_res_self = np.sum(resid_self**2) + 1e-8
        for j, cause in enumerate(names):
            if i == j:
                continue
            X_full = np.stack([regime_series[cause][max_lag-l:T-l] for l in range(1,max_lag+1)], axis=1)
            X_full = np.hstack([X_self, X_full])
            beta_full, *_ = lstsq(X_full, y, rcond=None)
            resid_full = y - X_full @ beta_full
            ss_res_full = np.sum(resid_full**2) + 1e-8
            delta_r2 = max(0.0, (ss_res_self - ss_res_full) / ss_res_self)
            adj[j, i] = delta_r2  # cause j -> effect i
    adj[adj < np.percentile(adj, 60)] = 0.0
    return adj

def compute_regime_causal_graphs(df, causal_vars=CAUSAL_VARS, max_lag=3):
    """Pre-compute regime-specific causal graphs: monsoon / inter-monsoon / heatwave.
    Graph is NOT rebuilt per batch/epoch -- only once here, cached and reused."""
    graphs = {}
    dates = pd.to_datetime(df["date"]) if "date" in df.columns else None
    regimes = {
        "monsoon": df["month"].isin([6,7,8,9]) if "month" in df.columns else None,
        "inter_monsoon": ~df["month"].isin([6,7,8,9]) if "month" in df.columns else None,
        "heatwave": df["srhi"] > df["srhi"].quantile(0.9) if "srhi" in df.columns else None,
    }
    for regime_name, mask in regimes.items():
        if mask is None:
            continue
        sub = df[mask]
        if len(sub) < 20:
            graphs[regime_name] = np.zeros((len(causal_vars), len(causal_vars)), dtype=np.float32)
            continue
        ts = sub.groupby("date")[causal_vars].mean().sort_index()
        series_dict = {c: ts[c].interpolate().bfill().ffill().values for c in causal_vars}
        graphs[regime_name] = granger_causal_adjacency(series_dict, max_lag=max_lag)
    return graphs

# ---------- 2.7 Graph encoder (lightweight GCN) producing conditioning embeddings ----------
class SimpleGCNEncoder(nn.Module):
    def __init__(self, n_nodes, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.n_nodes = n_nodes
        self.node_embed = nn.Parameter(torch.randn(n_nodes, in_dim) * 0.1)
        self.W1 = nn.Linear(in_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, adj):
        # adj: [n_nodes, n_nodes] numpy or tensor
        if not torch.is_tensor(adj):
            adj = torch.tensor(adj, dtype=torch.float32, device=self.node_embed.device)
        A = adj + torch.eye(self.n_nodes, device=adj.device)
        D = A.sum(dim=1, keepdim=True).clamp(min=1e-6)
        A_norm = A / D
        h = F.relu(self.W1(A_norm @ self.node_embed))
        h = self.W2(A_norm @ h)
        graph_embedding = h.mean(dim=0)  # pooled conditioning vector
        return h, graph_embedding  # node-level, pooled

def select_active_causal_graph(graphs_dict, current_month, current_srhi_high):
    if current_srhi_high:
        return graphs_dict.get("heatwave")
    if current_month in [6,7,8,9]:
        return graphs_dict.get("monsoon")
    return graphs_dict.get("inter_monsoon")

print("PART 2 loaded: physics residual + causal graph utilities ready.")


# PART 3 — Physics-Guided Causal Hybrid Architecture (ConvLSTM3D+CBAM+FNO+Event+Causal+Bayesian Heads)

In [ ]:
# ================= PART 3: HYBRID PHYSICS-GUIDED CAUSAL ARCHITECTURE =================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------- 3.1 ConvLSTM3D cell ----------
class ConvLSTM3DCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.hidden_ch = hidden_ch
        self.conv = nn.Conv3d(in_ch + hidden_ch, 4*hidden_ch, kernel_size, padding=pad)

    def forward(self, x, state):
        h, c = state
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_new = f*c + i*g
        h_new = o*torch.tanh(c_new)
        return h_new, c_new

    def init_state(self, batch, spatial_shape, device):
        D,H,W = spatial_shape
        h = torch.zeros(batch, self.hidden_ch, D, H, W, device=device)
        c = torch.zeros(batch, self.hidden_ch, D, H, W, device=device)
        return h, c

class ConvLSTM3D(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        self.cell = ConvLSTM3DCell(in_ch, hidden_ch, kernel_size)
        self.hidden_ch = hidden_ch

    def forward(self, x_seq):
        # x_seq: [B, T, C, D, H, W]
        B, T, C, D, H, W = x_seq.shape
        h, c = self.cell.init_state(B, (D,H,W), x_seq.device)
        outputs = []
        for t in range(T):
            h, c = self.cell(x_seq[:,t], (h,c))
            outputs.append(h)
        return torch.stack(outputs, dim=1), (h, c)

# ---------- 3.2 CBAM3D (multi-scale channel + spatial attention) ----------
class ChannelAttention3D(nn.Module):
    def __init__(self, ch, ratio=8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv3d(ch, ch//ratio, 1), nn.ReLU(),
            nn.Conv3d(ch//ratio, ch, 1))

    def forward(self, x):
        avg = F.adaptive_avg_pool3d(x, 1)
        mx = F.adaptive_max_pool3d(x, 1)
        att = torch.sigmoid(self.mlp(avg) + self.mlp(mx))
        return x * att

class SpatialAttention3D(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, kernel_size, padding=kernel_size//2)

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        att = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * att

class MultiScaleCBAM3D(nn.Module):
    def __init__(self, ch, scales=(1,2,4)):
        super().__init__()
        self.scales = scales
        self.ca = nn.ModuleList([ChannelAttention3D(ch) for _ in scales])
        self.sa = nn.ModuleList([SpatialAttention3D() for _ in scales])
        self.fuse = nn.Conv3d(ch*len(scales), ch, 1)

    def forward(self, x):
        outs = []
        for s, ca, sa in zip(self.scales, self.ca, self.sa):
            if s > 1:
                xs = F.avg_pool3d(x, kernel_size=(1,s,s), stride=(1,s,s))
            else:
                xs = x
            xs = ca(xs); xs = sa(xs)
            if s > 1:
                xs = F.interpolate(xs, size=x.shape[2:], mode="trilinear", align_corners=False)
            outs.append(xs)
        return self.fuse(torch.cat(outs, dim=1))

# ---------- 3.3 Fourier Neural Operator residual block (3D spectral conv) ----------
class SpectralConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, modes_d, modes_h, modes_w):
        super().__init__()
        self.in_ch, self.out_ch = in_ch, out_ch
        self.modes_d, self.modes_h, self.modes_w = modes_d, modes_h, modes_w
        scale = 1.0 / (in_ch*out_ch)
        self.weight = nn.Parameter(scale*torch.rand(
            in_ch, out_ch, modes_d, modes_h, modes_w, dtype=torch.cfloat))

    def compl_mul(self, inp, weights):
        return torch.einsum("bixyz,ioxyz->boxyz", inp, weights)

    def forward(self, x):
        B, C, D, H, W = x.shape
        x_ft = torch.fft.rfftn(x, dim=(-3,-2,-1))
        out_ft = torch.zeros(B, self.out_ch, D, H, W//2+1, dtype=torch.cfloat, device=x.device)
        md = min(self.modes_d, D)
        mh = min(self.modes_h, H)
        mw = min(self.modes_w, W//2+1)
        out_ft[:, :, :md, :mh, :mw] = self.compl_mul(
            x_ft[:, :, :md, :mh, :mw], self.weight[:, :, :md, :mh, :mw])
        x_out = torch.fft.irfftn(out_ft, s=(D,H,W), dim=(-3,-2,-1))
        return x_out

class FNOResidualBlock3D(nn.Module):
    def __init__(self, ch, modes=(4,8,8)):
        super().__init__()
        self.spec = SpectralConv3d(ch, ch, *modes)
        self.local = nn.Conv3d(ch, ch, 1)
        self.norm = nn.GroupNorm(min(8,ch), ch)

    def forward(self, x):
        y = self.spec(x) + self.local(x)
        return self.norm(F.gelu(y) + x)

# ---------- 3.4 Depth-regime multi-scale attention (surface / thermocline / DCM) ----------
class RegimeAttention(nn.Module):
    def __init__(self, ch, n_depth, surf_frac=0.25, dcm_frac=0.35):
        super().__init__()
        self.n_depth = n_depth
        self.n_surf = max(1, int(n_depth*surf_frac))
        self.n_dcm = max(1, int(n_depth*dcm_frac))
        self.attn_surf = nn.MultiheadAttention(ch, num_heads=4, batch_first=True)
        self.attn_thermo = nn.MultiheadAttention(ch, num_heads=4, batch_first=True)
        self.attn_dcm = nn.MultiheadAttention(ch, num_heads=4, batch_first=True)
        self.fuse = nn.Conv3d(ch, ch, 1)

    def _apply(self, x_slice, attn):
        B, C, d, H, W = x_slice.shape
        tok = x_slice.permute(0,2,3,4,1).reshape(B, d*H*W, C)
        out, _ = attn(tok, tok, tok)
        return out.reshape(B, d, H, W, C).permute(0,4,1,2,3)

    def forward(self, x):
        D = x.shape[2]
        s = self.n_surf; e_dcm = D - self.n_dcm
        surf = x[:, :, :s]
        thermo = x[:, :, s:e_dcm]
        dcm = x[:, :, e_dcm:]
        surf_o = self._apply(surf, self.attn_surf) if surf.shape[2]>0 else surf
        thermo_o = self._apply(thermo, self.attn_thermo) if thermo.shape[2]>0 else thermo
        dcm_o = self._apply(dcm, self.attn_dcm) if dcm.shape[2]>0 else dcm
        out = torch.cat([surf_o, thermo_o, dcm_o], dim=2)
        return self.fuse(out) + x

# ---------- 3.5 Event-Conditioning Module (FiLM injected into forward path) ----------
class EventConditioningFiLM(nn.Module):
    def __init__(self, event_dim, ch):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(event_dim, ch*2), nn.ReLU(), nn.Linear(ch*2, ch*2))

    def forward(self, x, event_vec):
        gamma_beta = self.mlp(event_vec)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        gamma = gamma.view(*gamma.shape, 1,1,1)
        beta = beta.view(*beta.shape, 1,1,1)
        return x * (1+gamma) + beta

def compute_event_vector(x_last_frame, feat_idx, prev_frame=None):
    # x_last_frame: [B, C, D, H, W] normalized features
    hw = x_last_frame[:, feat_idx["hwdepth"]].mean(dim=(1,2))
    n2 = x_last_frame[:, feat_idx["n2"]]
    n2_collapse = n2.std(dim=(1,2,3))
    if prev_frame is not None:
        no3_pulse = (x_last_frame[:, feat_idx["no3"]] - prev_frame[:, feat_idx["no3"]]).mean(dim=(1,2,3))
    else:
        no3_pulse = torch.zeros_like(n2_collapse)
    return torch.stack([hw, n2_collapse, no3_pulse], dim=-1)

# ---------- 3.6 Causal-graph conditioning (FiLM using pooled graph embedding) ----------
class CausalGraphFiLM(nn.Module):
    def __init__(self, graph_dim, ch):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(graph_dim, ch), nn.ReLU(), nn.Linear(ch, ch*2))

    def forward(self, x, graph_embed):
        graph_embed = graph_embed.unsqueeze(0).expand(x.shape[0], -1)
        gamma_beta = self.mlp(graph_embed)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        gamma = gamma.view(*gamma.shape, 1,1,1)
        beta = beta.view(*beta.shape, 1,1,1)
        return x * (1+gamma) + beta

# ---------- 3.7 Bayesian probabilistic output heads (MC Dropout + mean/var) ----------
class ProbabilisticHead(nn.Module):
    def __init__(self, ch, dropout=0.15):
        super().__init__()
        self.drop = nn.Dropout3d(dropout)
        self.mean_head = nn.Conv3d(ch, 1, 1)
        self.logvar_head = nn.Conv3d(ch, 1, 1)
        self.hab_prob_head = nn.Conv3d(ch, 1, 1)

    def forward(self, x):
        x = self.drop(x)
        mean = F.softplus(self.mean_head(x))
        logvar = self.logvar_head(x).clamp(-6, 6)
        hab_prob = torch.sigmoid(self.hab_prob_head(x))
        return mean, logvar, hab_prob

# ---------- 3.8 Full hybrid model ----------
class PhysicsGuidedCausalHybridModel(nn.Module):
    def __init__(self, in_ch, hidden_ch=32, n_depth=12, event_dim=3, graph_dim=16,
                 fno_modes=(4,8,8), use_fno=True, use_causal=True, use_event=True,
                 use_regime_attn=True, mc_dropout=0.15):
        super().__init__()
        self.use_fno = use_fno
        self.use_causal = use_causal
        self.use_event = use_event
        self.use_regime_attn = use_regime_attn

        self.convlstm = ConvLSTM3D(in_ch, hidden_ch)
        self.cbam = MultiScaleCBAM3D(hidden_ch)
        if use_fno:
            self.fno = FNOResidualBlock3D(hidden_ch, modes=fno_modes)
        if use_regime_attn:
            self.regime_attn = RegimeAttention(hidden_ch, n_depth)
        if use_event:
            self.event_film = EventConditioningFiLM(event_dim, hidden_ch)
        if use_causal:
            self.causal_film = CausalGraphFiLM(graph_dim, hidden_ch)
        self.proj = nn.Conv3d(hidden_ch, hidden_ch, 3, padding=1)
        self.head = ProbabilisticHead(hidden_ch, dropout=mc_dropout)
        self.physics_proj = None  # set from PART 2 if desired

    def forward(self, x_seq, event_vec=None, graph_embed=None):
        h_seq, (h_last, c_last) = self.convlstm(x_seq)
        h = h_last
        h = self.cbam(h)
        if self.use_fno:
            h = self.fno(h)
        if self.use_regime_attn:
            h = self.regime_attn(h)
        if self.use_event and event_vec is not None:
            h = self.event_film(h, event_vec)
        if self.use_causal and graph_embed is not None:
            h = self.causal_film(h, graph_embed)
        h = F.gelu(self.proj(h))
        mean, logvar, hab_prob = self.head(h)
        return mean.squeeze(1), logvar.squeeze(1), hab_prob.squeeze(1)

    def mc_predict(self, x_seq, event_vec=None, graph_embed=None, n_samples=20):
        self.train()  # keep dropout active for MC sampling
        preds = []
        with torch.no_grad():
            for _ in range(n_samples):
                mean, logvar, hab_prob = self.forward(x_seq, event_vec, graph_embed)
                preds.append(mean.unsqueeze(0))
        preds = torch.cat(preds, dim=0)
        mc_mean = preds.mean(dim=0)
        mc_std = preds.std(dim=0)
        return mc_mean, mc_std

# ---------- 3.9 Ablation variant factory ----------
def build_ablation_model(variant, in_ch, hidden_ch=32, n_depth=12):
    """variant in {'a_baseline','b_physics','c_fno','d_full'}
    physics residual is applied via loss (PART 2/4), not architecture,
    so 'a' and 'b' share architecture (attention-only), differing in loss terms."""
    if variant == "a_baseline":
        return PhysicsGuidedCausalHybridModel(in_ch, hidden_ch, n_depth,
                use_fno=False, use_causal=False, use_event=False, use_regime_attn=True)
    if variant == "b_physics":
        return PhysicsGuidedCausalHybridModel(in_ch, hidden_ch, n_depth,
                use_fno=False, use_causal=False, use_event=False, use_regime_attn=True)
    if variant == "c_fno":
        return PhysicsGuidedCausalHybridModel(in_ch, hidden_ch, n_depth,
                use_fno=True, use_causal=False, use_event=False, use_regime_attn=True)
    if variant == "d_full":
        return PhysicsGuidedCausalHybridModel(in_ch, hidden_ch, n_depth,
                use_fno=True, use_causal=True, use_event=True, use_regime_attn=True)
    raise ValueError(variant)

# ---------- 3.10 Simple Transformer baseline ----------
class TransformerBaseline(nn.Module):
    def __init__(self, in_ch, n_depth, n_lat, n_lon, d_model=64, nhead=4, n_layers=2):
        super().__init__()
        self.in_proj = nn.Linear(in_ch, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.out_proj = nn.Linear(d_model, 1)
        self.n_depth, self.n_lat, self.n_lon = n_depth, n_lat, n_lon

    def forward(self, x_seq):
        B, T, C, D, H, W = x_seq.shape
        x = x_seq.mean(dim=(3,4,5))  # [B,T,C] global pooled series baseline
        x = self.in_proj(x)
        x = self.encoder(x)
        out = self.out_proj(x[:, -1])
        return out.squeeze(-1)

print("PART 3 loaded: hybrid physics-guided causal architecture ready.")


# PART 4 — Training, Optuna Hyperparameter Optimization, Evaluation, Ablation, Cross-Regime/LOYO, Process Attribution

In [ ]:
# ================= PART 4: TRAINING, OPTUNA HPO, EVALUATION, ABLATION =================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
try:
    import optuna
except ImportError:
    optuna = None

# ---------- 4.1 Dataset ----------
class HABSequenceDataset(Dataset):
    def __init__(self, X, Y_dict, u_idx, v_idx, w_field, K_field, no3_idx, temp_idx,
                 mld_idx, event_extra=None):
        self.X = X
        self.Y_dict = Y_dict
        self.u_idx, self.v_idx = u_idx, v_idx
        self.w_field, self.K_field = w_field, K_field
        self.no3_idx, self.temp_idx, self.mld_idx = no3_idx, temp_idx, mld_idx

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = torch.tensor(self.X[i], dtype=torch.float32)
        y = {h: torch.tensor(self.Y_dict[h][i], dtype=torch.float32) for h in self.Y_dict}
        return x, y, i

def collate_batch(batch):
    xs = torch.stack([b[0] for b in batch])
    idxs = [b[2] for b in batch]
    y_out = {}
    for h in batch[0][1].keys():
        y_out[h] = torch.stack([b[1][h] for b in batch])
    return xs, y_out, idxs

# ---------- 4.2 Loss function ----------
def gaussian_nll(mean, logvar, target):
    inv_var = torch.exp(-logvar)
    return (0.5*inv_var*(target-mean)**2 + 0.5*logvar).mean()

def hab_bce_loss(hab_prob, target_chl, threshold=5.0):
    label = (target_chl > threshold).float()
    return F.binary_cross_entropy(hab_prob.clamp(1e-6,1-1e-6), label)

def composite_loss(mean, logvar, hab_prob, target, physics_res_loss=None,
                    stoich_loss=None, w_data=1.0, w_physics=0.1, w_stoich=0.05, w_hab=0.2):
    data_loss = gaussian_nll(mean, logvar, target)
    hab_loss = hab_bce_loss(hab_prob, target)
    total = w_data*data_loss + w_hab*hab_loss
    if physics_res_loss is not None:
        total = total + w_physics*physics_res_loss
    if stoich_loss is not None:
        total = total + w_stoich*stoich_loss
    return total, {"data": data_loss.item(), "hab": hab_loss.item(),
                    "physics": float(physics_res_loss) if physics_res_loss is not None else 0.0}

# ---------- 4.3 Training loop ----------
def train_model(model, train_loader, val_loader, cfg_model, epochs=30, lr=1e-3,
                 w_physics=0.1, w_stoich=0.05, device=DEVICE, use_physics=True,
                 event_provider=None, graph_embed=None, patience=6, verbose=True):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=3, factor=0.5)
    best_val, best_state, no_improve = np.inf, None, 0
    horizon = min(cfg_model.horizons)
    history = []
    for ep in range(epochs):
        model.train()
        train_losses = []
        for x, y, idxs in train_loader:
            x = x.to(device)
            target = y[horizon].to(device)
            event_vec = event_provider(x) if event_provider is not None else None
            g_embed = graph_embed.to(device) if graph_embed is not None else None
            mean, logvar, hab_prob = model(x, event_vec, g_embed)
            phys_loss = None
            if use_physics and hasattr(model, "physics_enabled") and model.physics_enabled:
                phys_loss = torch.tensor(0.0, device=device)
            loss, parts = composite_loss(mean[:, -1], logvar[:, -1], hab_prob[:, -1],
                                          target[:, -1] if target.dim()>3 else target,
                                          physics_res_loss=phys_loss, w_physics=w_physics)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            train_losses.append(loss.item())
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x, y, idxs in val_loader:
                x = x.to(device)
                target = y[horizon].to(device)
                event_vec = event_provider(x) if event_provider is not None else None
                g_embed = graph_embed.to(device) if graph_embed is not None else None
                mean, logvar, hab_prob = model(x, event_vec, g_embed)
                loss, _ = composite_loss(mean[:, -1], logvar[:, -1], hab_prob[:, -1],
                                          target[:, -1] if target.dim()>3 else target)
                val_losses.append(loss.item())
        tr_l, va_l = np.mean(train_losses), np.mean(val_losses)
        sched.step(va_l)
        history.append((tr_l, va_l))
        if verbose:
            print(f"epoch {ep+1}/{epochs} train={tr_l:.4f} val={va_l:.4f}")
        if va_l < best_val - 1e-4:
            best_val, best_state, no_improve = va_l, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose: print("early stopping")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

# ---------- 4.4 Evaluation metrics: CRPS, reliability, CRPSS ----------
def crps_gaussian(mean, std, target):
    z = (target - mean) / (std + 1e-6)
    phi = torch.exp(-0.5*z**2) / np.sqrt(2*np.pi)
    Phi = 0.5*(1+torch.erf(z/np.sqrt(2)))
    crps = std*(z*(2*Phi-1) + 2*phi - 1/np.sqrt(np.pi))
    return crps.mean().item()

def reliability_diagram_data(mean, std, target, n_bins=10):
    quantiles = np.linspace(0.05, 0.95, n_bins)
    from scipy.stats import norm
    empirical = []
    for q in quantiles:
        z = norm.ppf(q)
        pred_q = mean + z*std
        empirical.append((target <= pred_q).float().mean().item())
    return quantiles, empirical

def crpss(crps_model, crps_ref):
    return 1 - crps_model/(crps_ref+1e-8)

def rmse_mae(pred, target):
    pred, target = np.asarray(pred), np.asarray(target)
    rmse = np.sqrt(np.mean((pred-target)**2))
    mae = np.mean(np.abs(pred-target))
    return rmse, mae

# ---------- 4.5 Baselines: persistence, climatology ----------
def persistence_baseline(X_last_chl):
    return X_last_chl

def climatology_baseline(train_targets_by_doy, test_doy):
    return np.array([train_targets_by_doy.get(d, np.nanmean(list(train_targets_by_doy.values())))
                      for d in test_doy])

# ---------- 4.6 Optuna hyperparameter optimization ----------
def build_optuna_objective(train_loader, val_loader, in_ch, n_depth, cfg_model, device=DEVICE):
    def objective(trial):
        hidden_ch = trial.suggest_categorical("hidden_ch", [16, 32, 48])
        lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        w_physics = trial.suggest_float("w_physics", 0.01, 0.5, log=True)
        fno_mode = trial.suggest_categorical("fno_mode", [4, 6, 8])
        mc_dropout = trial.suggest_float("mc_dropout", 0.05, 0.3)
        model = PhysicsGuidedCausalHybridModel(
            in_ch, hidden_ch=hidden_ch, n_depth=n_depth,
            fno_modes=(fno_mode, fno_mode, fno_mode), mc_dropout=mc_dropout)
        model, history = train_model(model, train_loader, val_loader, cfg_model,
                                      epochs=8, lr=lr, w_physics=w_physics,
                                      device=device, verbose=False, patience=3)
        return history[-1][1]
    return objective

def run_optuna_search(train_loader, val_loader, in_ch, n_depth, cfg_model, n_trials=20):
    if optuna is None:
        print("optuna not installed; skipping HPO. pip install optuna --break-system-packages")
        return None
    study = optuna.create_study(direction="minimize")
    objective = build_optuna_objective(train_loader, val_loader, in_ch, n_depth, cfg_model)
    study.optimize(objective, n_trials=n_trials)
    print("Best params:", study.best_params)
    return study

# ---------- 4.7 Ablation study runner ----------
def run_ablation_study(X, Y, tr_idx, va_idx, te_idx, in_ch, n_depth, cfg_model,
                        epochs=15, device=DEVICE):
    results = {}
    variants = ["a_baseline", "b_physics", "c_fno", "d_full"]
    for variant in variants:
        model = build_ablation_model(variant, in_ch, hidden_ch=32, n_depth=n_depth)
        train_ds = HABSequenceDataset(X[tr_idx], {h: Y[h][tr_idx] for h in Y},
                                       0,0,None,None,0,0,0)
        val_ds = HABSequenceDataset(X[va_idx], {h: Y[h][va_idx] for h in Y},
                                     0,0,None,None,0,0,0)
        test_ds = HABSequenceDataset(X[te_idx], {h: Y[h][te_idx] for h in Y},
                                      0,0,None,None,0,0,0)
        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_batch)
        val_loader = DataLoader(val_ds, batch_size=4, collate_fn=collate_batch)
        test_loader = DataLoader(test_ds, batch_size=4, collate_fn=collate_batch)
        model, hist = train_model(model, train_loader, val_loader, cfg_model,
                                   epochs=epochs, use_physics=(variant!="a_baseline"),
                                   device=device, verbose=False)
        metrics = evaluate_model(model, test_loader, cfg_model, device=device)
        results[variant] = metrics
        print(variant, metrics)
    return results

# ---------- 4.8 Full evaluation with multi-horizon + MC uncertainty ----------
def evaluate_model(model, loader, cfg_model, device=DEVICE, n_mc=15):
    model.to(device)
    horizon = min(cfg_model.horizons)
    all_mean, all_std, all_target = [], [], []
    for x, y, idxs in loader:
        x = x.to(device)
        target = y[horizon].to(device)
        mc_mean, mc_std = model.mc_predict(x, n_samples=n_mc)
        all_mean.append(mc_mean[:, -1].reshape(x.shape[0], -1).mean(dim=1).cpu())
        all_std.append(mc_std[:, -1].reshape(x.shape[0], -1).mean(dim=1).cpu())
        t = target[:, -1] if target.dim()>3 else target
        all_target.append(t.reshape(x.shape[0], -1).mean(dim=1).cpu())
    mean = torch.cat(all_mean); std = torch.cat(all_std).clamp(min=1e-3)
    target = torch.cat(all_target)
    rmse, mae = rmse_mae(mean.numpy(), target.numpy())
    crps = crps_gaussian(mean, std, target)
    return {"rmse": rmse, "mae": mae, "crps": crps}

def multi_horizon_evaluation(model, X, Y, idx_set, cfg_model, device=DEVICE):
    out = {}
    for h in cfg_model.horizons:
        ds = HABSequenceDataset(X[idx_set], {h: Y[h][idx_set]}, 0,0,None,None,0,0,0)
        loader = DataLoader(ds, batch_size=4, collate_fn=collate_batch)
        cfg_h = cfg_model
        out[h] = evaluate_model(model, loader, cfg_model, device=device)
    return out

# ---------- 4.9 Cross-regime & leave-one-year-out evaluation ----------
def cross_regime_evaluation(model, X, Y, seq_dates, cfg_model, device=DEVICE):
    regimes = get_regime_labels(seq_dates)
    results = {}
    for r in np.unique(regimes):
        idx = np.where(regimes == r)[0]
        if len(idx) < 4: continue
        ds = HABSequenceDataset(X[idx], {h: Y[h][idx] for h in Y}, 0,0,None,None,0,0,0)
        loader = DataLoader(ds, batch_size=4, collate_fn=collate_batch)
        results[r] = evaluate_model(model, loader, cfg_model, device=device)
    return results

def loyo_evaluation(build_model_fn, X, Y, seq_dates, cfg_model, epochs=10, device=DEVICE):
    results = {}
    for test_year, train_mask, test_mask in leave_one_year_out_split(seq_dates):
        tr_idx = np.where(train_mask)[0]
        te_idx = np.where(test_mask)[0]
        if len(te_idx) < 4 or len(tr_idx) < 8:
            continue
        n_val = max(2, int(0.15*len(tr_idx)))
        va_idx, tr_idx2 = tr_idx[:n_val], tr_idx[n_val:]
        model = build_model_fn()
        train_ds = HABSequenceDataset(X[tr_idx2], {h: Y[h][tr_idx2] for h in Y}, 0,0,None,None,0,0,0)
        val_ds = HABSequenceDataset(X[va_idx], {h: Y[h][va_idx] for h in Y}, 0,0,None,None,0,0,0)
        test_ds = HABSequenceDataset(X[te_idx], {h: Y[h][te_idx] for h in Y}, 0,0,None,None,0,0,0)
        train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_batch)
        val_loader = DataLoader(val_ds, batch_size=4, collate_fn=collate_batch)
        test_loader = DataLoader(test_ds, batch_size=4, collate_fn=collate_batch)
        model, _ = train_model(model, train_loader, val_loader, cfg_model, epochs=epochs,
                                device=device, verbose=False)
        results[test_year] = evaluate_model(model, test_loader, cfg_model, device=device)
        print("LOYO year", test_year, results[test_year])
    return results

# ---------- 4.10 Extreme-event holdout ----------
def extreme_event_holdout_evaluation(model, X, Y, seq_dates, srhi_series, cfg_model,
                                      quantile=0.9, device=DEVICE):
    thresh = np.quantile(srhi_series, quantile)
    extreme_idx = np.where(srhi_series >= thresh)[0]
    if len(extreme_idx) < 4:
        return {}
    ds = HABSequenceDataset(X[extreme_idx], {h: Y[h][extreme_idx] for h in Y}, 0,0,None,None,0,0,0)
    loader = DataLoader(ds, batch_size=4, collate_fn=collate_batch)
    return evaluate_model(model, loader, cfg_model, device=device)

# ---------- 4.11 Perturbation-based process attribution ----------
def process_attribution(model, X_sample, Y_sample, feat_idx, cfg_model, device=DEVICE):
    model.eval()
    x = torch.tensor(X_sample, dtype=torch.float32, device=device)
    with torch.no_grad():
        base_mean, _, _ = model(x)
        base_pred = base_mean[:, -1].mean().item()
    attributions = {}
    for var, ci in feat_idx.items():
        x_pert = x.clone()
        x_pert[:, :, ci] = 0.0
        with torch.no_grad():
            pert_mean, _, _ = model(x_pert)
            pert_pred = pert_mean[:, -1].mean().item()
        attributions[var] = abs(base_pred - pert_pred)
    return dict(sorted(attributions.items(), key=lambda kv: -kv[1]))

def attention_and_causal_edge_summary(model, causal_graph, causal_vars):
    edge_strength = {}
    n = len(causal_vars)
    for i in range(n):
        for j in range(n):
            if causal_graph[i, j] > 0:
                edge_strength[f"{causal_vars[i]}->{causal_vars[j]}"] = float(causal_graph[i, j])
    return dict(sorted(edge_strength.items(), key=lambda kv: -kv[1]))

# ---------- 4.12 Plot helpers ----------
def plot_reliability(quantiles, empirical, title="Reliability Diagram"):
    plt.figure(figsize=(4,4))
    plt.plot([0,1],[0,1],"k--")
    plt.plot(quantiles, empirical, "o-")
    plt.xlabel("Nominal quantile"); plt.ylabel("Empirical coverage")
    plt.title(title); plt.tight_layout(); plt.show()

def plot_training_history(history, title="Training/Validation Loss"):
    tr = [h[0] for h in history]; va = [h[1] for h in history]
    plt.figure(figsize=(5,3))
    plt.plot(tr, label="train"); plt.plot(va, label="val")
    plt.legend(); plt.title(title); plt.tight_layout(); plt.show()

print("PART 4 loaded: training / Optuna HPO / evaluation / ablation pipeline ready.")

# ================= END-TO-END DRIVER (optional) =================
if __name__ == "__main__":
    df = load_raw_dataframe(CSV_PATH)
    cfg = GridConfig(df)
    tensor, dates = build_full_spatiotemporal_tensor(df, cfg)
    tensor_norm, feat_mean, feat_std = normalize_tensor(tensor)
    X, Y, seq_dates = make_sequences(tensor_norm, dates, cfg, FEAT_IDX["chl"])
    tr_idx, va_idx, te_idx = train_val_test_split_by_time(len(X))
    in_ch = len(FEATURE_VARS)

    causal_graphs = compute_regime_causal_graphs(df)
    gcn = SimpleGCNEncoder(n_nodes=len(CAUSAL_VARS), in_dim=8, hidden_dim=16, out_dim=16)
    active_graph = select_active_causal_graph(causal_graphs, current_month=7, current_srhi_high=False)
    _, graph_embed = gcn(active_graph if active_graph is not None else
                          np.zeros((len(CAUSAL_VARS),len(CAUSAL_VARS)),dtype=np.float32))

    train_ds = HABSequenceDataset(X[tr_idx], {h: Y[h][tr_idx] for h in Y}, 0,0,None,None,0,0,0)
    val_ds = HABSequenceDataset(X[va_idx], {h: Y[h][va_idx] for h in Y}, 0,0,None,None,0,0,0)
    test_ds = HABSequenceDataset(X[te_idx], {h: Y[h][te_idx] for h in Y}, 0,0,None,None,0,0,0)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_ds, batch_size=4, collate_fn=collate_batch)
    test_loader = DataLoader(test_ds, batch_size=4, collate_fn=collate_batch)

    study = run_optuna_search(train_loader, val_loader, in_ch, cfg.n_depth, cfg, n_trials=10)
    best_params = study.best_params if study is not None else {"hidden_ch":32,"lr":1e-3,"w_physics":0.1,"fno_mode":6,"mc_dropout":0.15}

    final_model = PhysicsGuidedCausalHybridModel(
        in_ch, hidden_ch=best_params["hidden_ch"], n_depth=cfg.n_depth,
        fno_modes=(best_params["fno_mode"],)*3, mc_dropout=best_params["mc_dropout"])
    final_model, history = train_model(final_model, train_loader, val_loader, cfg,
                                        epochs=30, lr=best_params["lr"],
                                        w_physics=best_params["w_physics"])
    plot_training_history(history)

    ablation_results = run_ablation_study(X, Y, tr_idx, va_idx, te_idx, in_ch, cfg.n_depth, cfg)
    mh_results = multi_horizon_evaluation(final_model, X, Y, te_idx, cfg)
    regime_results = cross_regime_evaluation(final_model, X, Y, seq_dates[te_idx], cfg)
    loyo_results = loyo_evaluation(lambda: PhysicsGuidedCausalHybridModel(in_ch, 32, cfg.n_depth),
                                    X, Y, seq_dates, cfg, epochs=10)
    attributions = process_attribution(final_model, X[te_idx][:8], Y[min(cfg.horizons)][te_idx][:8], FEAT_IDX, cfg)
    print("Process attribution:", attributions)
    print("Multi-horizon:", mh_results)
    print("Cross-regime:", regime_results)
